GPT2 implementation

- exploring hyperconnections and Deepseek mHC
- pre- or post-LayerNorm


In [ ]:
# normal dot product attention
import torch
import numpy as np
import torch.nn.functional as F

def DotProductAttention(Q, K, V):

	"""
	Args
	- Q = matrix of queries
	- K = matrix of keys
	- V = matrix of values

	Reasoning about time complexity: 
	- (b, t, t) * (b, t, d) -> (b, t, d), summing t rows of t terms with a column of t terms (t sums), over d columns
	- time complexity = t^2 * d

	Time complexity: O(batch * seq_len^2 * dim + batch * seq_len * dim)
	Space complexity: O(batch * seq_len * d + batch * seq_len^2) # 3 qkv matrices, then the attention matrix - removing constants
	"""

	b, t, d = Q.shape
	qk = torch.einsum('bqd, bkd -> bqk', Q, K) # 
	softmax_qk = F.softmax(qk / (d**0.5), dim=-1) # remember you need to softmax along the dim

	# causal masking
	mask = torch.tril(torch.ones(t,t)).bool()
	print(mask)
	masked_qk = softmax_qk.masked_fill(~mask, float('-inf'))

	qkv = torch.einsum('bqk, bkd -> bqd', softmax_qk, V)

	print(qkv.shape)
	return qkv

Q = torch.rand(32, 1024, 64)
K = torch.rand(32, 1024, 64)
V = torch.rand(32, 1024, 64)

torch.allclose(DotProductAttention(Q,K,V), F.scaled_dot_product_attention(Q, K, V))

tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True, False, False],
        [ True,  True,  True,  ...,  True,  True, False],
        [ True,  True,  True,  ...,  True,  True,  True]])
torch.Size([32, 1024, 64])


True

In [33]:
# multi-head attention

import torch
import numpy as np
import torch.nn.functional as F

def MultiHeadAttention(Q, K, V, n):

	"""
	Args
	- Q = matrix of queries
	- K = matrix of keys
	- V = matrix of values
	- n = number of heads for query, key and values

	In multi-head attention we want to split the query matrix into multiple heads, and same for the rest, and just multiply heads together
	"""
	B, T, D = Q.shape	
	assert D % n == 0

	chunked_Q = torch.split(Q, D//n, dim=-1)
	chunked_K = torch.split(K, D//n, dim=-1)
	chunked_V = torch.split(V, D//n, dim=-1)
	
	for Q, K, V in zip(chunked_Q, chunked_K, chunked_V):

		b, t, d = chunked_Q[0].shape
		qk = torch.einsum('bqd, bkd -> qkd', Q, K)
		softmax_qk = F.softmax(qk / (d**0.5), dim=-1)

		# causal masking
		mask = torch.tril(torch.ones(t,t)).bool()
		print(mask)
		masked_qk = softmax_qk.masked_fill(~mask, float('-inf'))

		qkv = torch.einsum('bvt, bvd -> btd', softmax_qk, V)

		print(qkv.shape)
		return qkv

Q = torch.rand(32, 1024, 64)
K = torch.rand(32, 1024, 64)
V = torch.rand(32, 1024, 64)

torch.allclose(MultiHeadAttention(Q,K,V, 8), F.scaled_dot_product_attention(Q, K, V))

tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True, False, False],
        [ True,  True,  True,  ...,  True,  True, False],
        [ True,  True,  True,  ...,  True,  True,  True]])


RuntimeError: The size of tensor a (1024) must match the size of tensor b (8) at non-singleton dimension 2

Normalization techniques

- batchnorm
- layernorm
- instancenorm
- groupnorm
- RMSnorm
